# Transfer Learning (Aprendizaje por Transferencia) 

Es una de las técnicas más potentes en la práctica, ya que permite aprovechar el conocimiento de modelos gigantes entrenados en millones de imágenes (como ImageNet) para resolver tus problemas con pocos datos.

## Importación de librerías

Primero, importamos PyTorch y las herramientas necesarias para manejar datos y visión por computador.

In [8]:
# Para instalars la librería timm (PyTorch Image Models), que tiene una colección enorme de modelos SOTA (State of the Art)
# se debe descomentar la siguiente linea
#!pip install timm

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import timm # Importamos la librería nueva
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


## Preparación de Datos (CIFAR-10)
Para este ejemplo, usaremos CIFAR-10, el cual contiene son imágenes a color y más complejas (aviones, pájaros, ranas), donde los modelos preentrenados en ImageNet brillan mucho más.

*Nota Importante:* Los modelos preentrenados (como ResNet) suelen esperar imágenes de tamaño 224x224. CIFAR tiene 32x32. Haremos un Resize en las transformaciones (aunque esto hace el entrenamiento más lento, es necesario para usar los pesos originales de ImageNet correctamente).

In [9]:
BATCH_SIZE = 256 #32 # Bajamos un poco el batch size porque las imágenes serán más grandes (224x224)

# Transformaciones:
# 1. Resize: Escalamos a 224x224 (estándar de ImageNet)
# 2. ToTensor: Convertimos a tensores
# 3. Normalize: Usamos la media y desviación estándar de ImageNet (necesario para Transfer Learning)
imagenet_stats = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

transform = transforms.Compose([
    transforms.Resize(224), 
    transforms.ToTensor(),
    transforms.Normalize(*imagenet_stats)
])

# Descargar datos (CIFAR-10)
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_dataset.classes
print(f"Clases: {class_names}")

Clases: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Enfoque 1 - Usando torchvision (Feature Extraction)
Aquí usaremos una ResNet18 clásica.

1. Cargamos el modelo con pesos preentrenados.

2. Congelamos los pesos (para aprovechar lo que la red ya sabe).

3. Reemplazamos la capa final (la "cabeza") para que clasifique 10 clases en lugar de las 1000 de ImageNet.

In [10]:
def get_torchvision_model():
    print("Cargando ResNet18 desde Torchvision...")
    
    # 1. Cargar modelo con pesos 'DEFAULT' (los mejores disponibles)
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    # 2. Congelar los parámetros (Feature Extraction)
    # Esto evita que se actualicen los pesos de las capas convolucionales
    for param in model.parameters():
        param.requires_grad = False
        
    # 3. Reemplazar la capa final (Fully Connected)
    # model.fc.in_features nos dice cuántas neuronas entran a la última capa
    num_ftrs = model.fc.in_features
    
    # Creamos una nueva capa lineal (que sí se entrenará) para nuestras 10 clases
    model.fc = nn.Linear(num_ftrs, 10)
    
    return model

model_tv = get_torchvision_model().to(device)

Cargando ResNet18 desde Torchvision...


## Enfoque 2 - Usando timm (Modelos Modernos)
timm hace esto mucho más fácil. Vamos a cargar un EfficientNet_B0, un modelo muy eficiente y potente. timm permite cambiar la capa final automáticamente pasando el argumento num_classes.

In [11]:
def get_timm_model():
    print("Cargando EfficientNet_B0 desde TIMM...")
    
    # timm.create_model hace todo el trabajo sucio
    # pretrained=True: Descarga los pesos
    # num_classes=10: Ajusta automáticamente la capa final
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=10)
    
    # Opcional: Congelar pesos (igual que arriba)
    # A veces en timm queremos hacer 'Fine Tuning' completo, pero congelaremos para comparar
    for param in model.parameters():
        param.requires_grad = False
        
    # En EfficientNet la capa final se llama 'classifier', no 'fc'.
    # timm ya la reemplazó por nosotros al poner num_classes=10, 
    # pero debemos asegurarnos de desbloquearla para entrenamiento.
    for param in model.get_classifier().parameters():
        param.requires_grad = True
        
    return model

model_timm = get_timm_model().to(device)

Cargando EfficientNet_B0 desde TIMM...


## Entrenamiento
Usaremos el mismo ciclo de entrenamiento que en los notebooks anteriores

In [12]:
# Función de entrenamiento estándar
def train_model(model, train_loader, test_loader, epochs=3):
    criterion = nn.CrossEntropyLoss()
    # Solo optimizamos los parámetros que requieren gradiente (la capa final)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
    
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        model.train()
        running_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        # Evaluación rápida al final de la época
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        print(f"Pérdida: {running_loss/len(train_loader):.4f} | Accuracy Test: {100 * correct / total:.2f}%")

## Ejecución y Comparación
Entrenamos ambos modelos por pocas épocas (al usar Transfer Learning, a veces 1 o 2 épocas bastan para obtener >80% de precisión).

In [13]:
print("--- Entrenando Modelo Torchvision (ResNet18) ---")
train_model(model_tv, train_loader, test_loader, epochs=5)

print("\n--- Entrenando Modelo TIMM (EfficientNet) ---")
train_model(model_timm, train_loader, test_loader, epochs=5)

--- Entrenando Modelo Torchvision (ResNet18) ---
Epoch 1/5
Pérdida: 1.0070 | Accuracy Test: 77.26%
Epoch 2/5
Pérdida: 0.6519 | Accuracy Test: 78.99%
Epoch 3/5
Pérdida: 0.6009 | Accuracy Test: 79.63%
Epoch 4/5
Pérdida: 0.5788 | Accuracy Test: 79.91%
Epoch 5/5
Pérdida: 0.5623 | Accuracy Test: 80.44%

--- Entrenando Modelo TIMM (EfficientNet) ---
Epoch 1/5
Pérdida: 1.7981 | Accuracy Test: 64.53%
Epoch 2/5
Pérdida: 0.9508 | Accuracy Test: 71.53%
Epoch 3/5
Pérdida: 0.7950 | Accuracy Test: 74.47%
Epoch 4/5
Pérdida: 0.7159 | Accuracy Test: 75.90%
Epoch 5/5
Pérdida: 0.6622 | Accuracy Test: 76.77%


## Listar modelos disponibles en TIMM
Si deseamos listar los modelos existentes dentro del módulo timm podemos usar la función `list_models` de TIMM. En el siguiente ejemplo listamos los modelos que tengan como base una arquitectura resnet:

In [14]:
# Listar todos los modelos disponibles en timm que contengan la palabra 'resnet'
all_resnets = timm.list_models('resnet*', pretrained=True)
print(f"Modelos ResNet disponibles en TIMM: {len(all_resnets)}")
print(all_resnets[:5]) # Imprimir los primeros 5

Modelos ResNet disponibles en TIMM: 106
['resnet10t.c3_in1k', 'resnet14t.c3_in1k', 'resnet18.a1_in1k', 'resnet18.a2_in1k', 'resnet18.a3_in1k']
